# 실험3 — carry 전달이 TE 위에서 값어치가 있나 (해준 담당)

**교수님 지시**: 우리 모델은 overlap 켜면 step size 때문에 TE를 못 붙이지만, **overlap 끄고 carry 전달만**
살리면 TE를 붙일 수 있다. 그래서 `bimamba + carry + TE` 가 `bimamba + TE`(carry 없음)보다 나은지 확인.
**비슷하면 carry도 제거** 검토.

- **재학습 없음**: 같은 `bimamba+carry` 체크포인트에 eval-time 플래그만 다르게 — `sscp_enabled` on/off + TE.
- overlap OFF (TE와 공존 불가). lr 1e-5(고정). K=100(=`bimamba` 폴더) 우선, 은지님 K=20/50 나오면 그때 재실행.
- 코드 확인함: TE 경로에서도 `_predict_with_carry`가 매 스텝 호출되어 carry가 실제 작동(on/off 결과 다름).


## 0) 부팅 + 소스 체크포인트 확인


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

# ── 무엇을 보나 ────────────────────────────────────────────────────────────
# 교수님 지시: overlap 끄고 carry 전달만 살린 채 TE 붙였을 때, 그냥 bimamba+TE 보다 나은가?
#   비슷하면 carry 도 제거. → carry(전달) on/off × TE, 둘 다 overlap OFF. 재학습 없음(eval-time 플래그만).
#
# ★ 소스 체크포인트 = bimamba+carry 학습본. 서버 폴더 'bimamba'(K=100)가 기본.
#   은지님 K=20/50 bimamba+carry 체크포인트 나오면 BASE_TAG/폴더만 바꿔 같은 셀 재실행.
BASE_TAG = 'bimamba'      # bimamba + carry(sscp) 학습본 (config 에 sscp_enabled=true). K=100.
v23.MODEL_DIR_NAMES.setdefault(BASE_TAG, BASE_TAG)

TASK  = 'libero_10'
SEEDS = [0, 1]            # (은지님은 seed2 사용 — 필요시 [2] 로)
N_EP  = 100              # task당 100 (overall 1000). 빠르게 보려면 50.
GPUS  = v23.available_gpus()

# TE(0.01) + n_action_steps=1. carry 전달만 on/off 로 대비.
_TE = ['--policy.temporal_ensemble_coeff=0.01', '--policy.n_action_steps=1']
# (overlap 은 소스가 bimamba(overlap 없는 정책)라 애초에 꺼져 있음 = 조건 충족)
VARIANTS = [
    ('bimamba_carry_te',   ['--policy.sscp_enabled=true']  + _TE),   # carry 전달 살림 + TE  (우리 주장)
    ('bimamba_nocarry_te', ['--policy.sscp_enabled=false'] + _TE),   # carry 끔 + TE          (baseline)
]
for vt, _ in VARIANTS:
    v23.MODEL_DIR_NAMES.setdefault(vt, vt)

print('소스(carry 학습본):', BASE_TAG, '| task', TASK, '| seeds', SEEDS, '| N_EP', N_EP, '| GPU', GPUS)
for s in SEEDS:
    cd = v23.best_ckpt_dir(BASE_TAG, s, TASK, how=cf.CKPT_STEP)
    print(f'   seed{s} 체크포인트:', '있음' if cd else '❌ 없음 (BASE_TAG/폴더 확인)')

## 1) eval (carry+TE / nocarry+TE)


In [ ]:
# carry+TE / nocarry+TE 두 변형을 같은 체크포인트에서 eval (재학습 X). action 기록 → 떨림 측정.
#   이미 유효 eval(overall n_ep>=N_EP*10/2) 있으면 skip. 끊기면 재실행 시 task 단위로 이어서.
_MINEP = 10 * N_EP // 2
def _has_valid(vt, s):
    info = v23.eval_clean_dir(vt, s, TASK) / 'eval_info.json'
    if not info.exists(): return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

jobs = [(vt, s, flags) for s in SEEDS for vt, flags in VARIANTS if not _has_valid(vt, s)]
print(f'실행 {len(jobs)} / 전체 {len(SEEDS)*len(VARIANTS)} (유효 완료 skip)')
ng = len(GPUS)
for i in range(0, len(jobs), ng):
    chunk = jobs[i:i + ng]
    labeled = []
    for g, (vt, s, flags) in zip(GPUS, chunk):
        try:
            cmd = v23.make_eval_cmd(BASE_TAG, seed=s, task=TASK, gpu_id=g, n_episodes=N_EP,
                                    select=cf.CKPT_STEP, extra_policy=flags,
                                    out_dir=v23.eval_clean_dir(vt, s, TASK))
            labeled.append((f'{vt}/seed{s}', cmd))
        except FileNotFoundError as e:
            print('  skip:', e)
    if labeled:
        print(f'\n===== eval 청크 {i//ng+1} ({len(labeled)} run) =====')
        v23.launch_cmds_live(labeled)
print('\n완료')

## 2) 결과 — SR + 떨림


In [ ]:
# 결과: carry+TE vs nocarry+TE (+ 참고: bimamba=carry,noTE / bimamba_te=carry+TE). SR + 떨림.
import numpy as np, smooth_metrics_paper as smp
importlib.reload(smp)
FS, STRIDE = cf.fps_of(TASK), 100
ROWS = [('bimamba_carry_te', 'bimamba + carry + TE  (우리)'),
        ('bimamba_nocarry_te', 'bimamba + TE (no carry)'),
        ('bimamba', 'bimamba (carry, TE 없음, 참고)')]
def rec(tag, s):
    d = cf.OUTPUT_BASE / 'eval_clean' / TASK / v23.MODEL_DIR_NAMES.get(tag, tag) / f'seed{s}'
    if not d.is_dir(): return None
    best = None
    for info in d.rglob('eval_info.json'):
        try: ov = json.loads(info.read_text()).get('overall', {})
        except Exception: continue
        n = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or n > best['n']:
            best = {'sr': ov.get('pc_success'), 'n': n, 'act': (info.parent/'actions').is_dir(), 'p': info.parent}
    return best

print('== SR ==')
print(f'{"model":<32}' + ''.join(f'{("s"+str(s)):>8}' for s in SEEDS) + f'{"mean":>8}')
for tag, lb in ROWS:
    vals = []
    cells = ''
    for s in SEEDS:
        e = rec(tag, s); sr = e['sr'] if (e and (e['n'] or 0) >= 10*N_EP//2) else None
        cells += (f'{sr:>8.1f}' if sr is not None else f'{chr(32)*8}')
        if sr is not None: vals.append(sr)
    m = f'{np.mean(vals):>8.1f}' if vals else f'{chr(32)*8}'
    print(f'{lb:<32}{cells}{m}')

print('\n== 떨림 (aloha 방식) ==')
print(f'{"model":<32}{"jerk":>9}{"bnd":>9}{"int":>9}{"B/I":>7}{"SPARC":>9}{"sflip":>9}{"n":>6}')
for tag, lb in ROWS:
    trajs = []
    for s in SEEDS:
        e = rec(tag, s)
        if e and (e['n'] or 0) >= 10*N_EP//2 and e['act']:
            t = v23._load_action_trajs(e['p']/'actions') or []
            if len(t) >= 80: trajs += t
    a = smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS) if trajs else None
    if a:
        print(f'{lb:<32}{a["jerk_rms_mean"]:>9.4f}{a["boundary_jerk_rms_mean"]:>9.4f}'
              f'{a["interior_jerk_rms_mean"]:>9.4f}{a["boundary_interior_ratio_mean"]:>7.2f}'
              f'{a["sparc_mean"]:>9.2f}{a["sign_flip_rate_mean"]:>9.4f}{a["n_traj"]:>6}')
    else:
        print(f'{lb:<32}' + ' '*49 + '(빈칸)')
print('\n판단: bimamba_carry_te(SR·떨림) 가 bimamba_nocarry_te 보다 확실히 나으면 carry 생존, 비슷하면 carry 제거 검토.')